# 🎫 LangChain Agent + MCP: IT Support Ticket Desk

This notebook builds a **custom LangChain agent** that uses our **IT Support Ticket MCP server**. The server is built with FastMCP and served over Streamable HTTP.

```
┌──────────────┐    ┌─────────────────────┐    ┌──────────────────────┐   HTTP    ┌──────────────────────┐    ┌────────┐
│  Notebook    │ ─► │  LangChain Agent    │ ─► │ MultiServerMCPClient │ ────────► │  FastMCP Server      │ ─► │ SQLite │
│  (you)       │    │  OpenAI + memory    │    │ (langchain-mcp-      │  /mcp     │  tools · resources · │    │        │
│              │ ◄─ │  + system prompt    │ ◄─ │  adapters)           │ ◄──────── │  prompts             │ ◄─ │        │
└──────────────┘    └─────────────────────┘    └──────────────────────┘           └──────────────────────┘    └────────┘
```

**How the three MCP primitives are used here**

| Primitive | Who uses it | In this notebook |
|---|---|---|
| **Tools** | The **model** decides when to call them | `register_ticket`, `search_tickets`, `get_ticket`, `update_ticket_status`, `resolve_ticket` |
| **Resources** | The **application** loads them as context | `tickets://stats`, `tickets://reference`, `tickets://{ticket_id}` |
| **Prompts** | The **user/application** picks a template | `triage_ticket`, `summarize_open` |

**Use cases**

1. **Help-desk intake**: turn a plain-English complaint into a ticket.
2. **Triage & work a ticket**: find, triage, start and resolve a P1 ticket.
3. **Manager status report**: an executive summary built from live stats.

> **Before you start:** the MCP server must be running (`make run` or `make up`), and `.env` must contain `OPENAI_API_KEY` and `AGENT_MODEL`.

## 1. Setup: configuration and server health check

In [ ]:
import json
import os
import re
import urllib.request
from datetime import datetime, timezone
from urllib.parse import urlsplit

from dotenv import find_dotenv, load_dotenv
from IPython.display import Markdown, display

load_dotenv(find_dotenv(usecwd=True))

MCP_SERVER_URL = os.getenv("MCP_SERVER_URL", "http://localhost:8989/mcp")
AGENT_MODEL = os.getenv("AGENT_MODEL", "gpt-5.4-mini")

assert os.getenv("OPENAI_API_KEY"), "OPENAI_API_KEY is missing - add it to your .env file"

# Health check: the server exposes GET /health next to the /mcp endpoint
parts = urlsplit(MCP_SERVER_URL)
health_url = f"{parts.scheme}://{parts.netloc}/health"
try:
    with urllib.request.urlopen(health_url, timeout=5) as resp:
        health = json.load(resp)
except Exception as exc:
    raise RuntimeError(
        f"MCP server is not reachable at {health_url}. Start it with `make run` or `make up`."
    ) from exc

print(f"✅ MCP server : {MCP_SERVER_URL}  ->  {health}")
print(f"✅ LLM model  : {AGENT_MODEL}")
print("✅ OpenAI key : loaded (not shown)")

## 2. Connect to the MCP server and load its tools

`MultiServerMCPClient` can connect to several MCP servers at once. We register a single server called `"tickets"` using the **Streamable HTTP** transport.

`get_tools()` turns every MCP tool into a LangChain `BaseTool`, keeping its name, description and JSON schema, so the LLM knows how to call it.

In [ ]:
from langchain_mcp_adapters.client import MultiServerMCPClient

client = MultiServerMCPClient(
    {
        "tickets": {
            "transport": "streamable_http",
            "url": MCP_SERVER_URL,
        }
    }
)

tools = await client.get_tools()

rows = "\n".join(f"| `{t.name}` | {t.description.splitlines()[0]} |" for t in tools)
display(Markdown(f"**{len(tools)} tools loaded from MCP**\n\n| Tool | Description |\n|---|---|\n{rows}"))

## 3. Explore Resources and Prompts

Unlike tools, the LLM does **not** call **resources** and **prompts** by itself. The application reads them and decides how to use them:

- `get_resources(server, uris=[...])` returns LangChain `Blob` objects.
- `get_prompt(server, name, arguments={...})` returns LangChain messages, ready to send to a model.

In [ ]:
# Resources -> Blob objects
blobs = await client.get_resources("tickets", uris=["tickets://reference", "tickets://stats"])
for blob in blobs:
    print(f"📄 {blob.metadata['uri']}")
    print(json.dumps(json.loads(blob.as_string()), indent=2), "\n")

In [ ]:
# Prompts -> LangChain messages
triage_messages = await client.get_prompt("tickets", "triage_ticket", arguments={"ticket_id": "TKT-10001"})
print(type(triage_messages[0]).__name__, "\n")
print(triage_messages[0].content[:700], "...")

## 4. Build the custom agent

We customise the agent in two ways:

1. **System prompt**: a help-desk persona with rules for choosing product family and priority (P1 to P4), and guardrails such as *never invent ticket IDs*.
2. **Conversation memory**: `InMemorySaver` stores each conversation under a `thread_id`, so follow-up questions keep their context.

In [ ]:
from langchain.agents import create_agent
from langchain_openai import ChatOpenAI
from langgraph.checkpoint.memory import InMemorySaver

TODAY = datetime.now(timezone.utc).strftime("%Y-%m-%d")

SYSTEM_PROMPT = f'''
You are **Ticket Desk**, a friendly and precise IT service-desk agent for a network team.
Today's date (UTC) is {TODAY}.

You manage IT support tickets ONLY through the provided tools. Ticket IDs look like TKT-10001.

## Registering tickets
- Infer the product family from the description: Routers, Switches, SD-WAN, Firewalls or Optic Cables.
  If it is genuinely unclear, ask ONE short clarifying question instead of guessing.
- Choose the priority:
  - P1 = outage / many users or a critical service down
  - P2 = major degradation or an important service impaired
  - P3 = minor issue, workaround exists, single user or site
  - P4 = request, question, planned change
- Write a concise title (max ~10 words) and a clear description. Use the reporter's name as registered_by.
  If the name is unknown, ask for it.

## Working tickets
- Lifecycle: Open -> In Progress -> Resolved -> Closed (Resolved can be reopened to Open).
- Use resolve_ticket to resolve (needs resolved_by and remarks). Use update_ticket_status for other changes.
- Never close a ticket unless the user explicitly asks.

## Answering
- NEVER invent tickets, IDs, names or numbers. Use the tools to look things up.
- Always mention ticket IDs in **bold**. Prefer short answers, bullet points and small markdown tables.
'''.strip()

model = ChatOpenAI(model=AGENT_MODEL)
memory = InMemorySaver()

agent = create_agent(
    model=model,
    tools=tools,
    system_prompt=SYSTEM_PROMPT,
    checkpointer=memory,
)
print("🤖 Agent ready with", len(tools), "MCP tools")

### Helper: `ask()`

`ask()` sends a message, or a list of messages, to the agent on a given **thread**. It then shows which MCP tools the agent called, followed by the final answer.

In [ ]:
from langchain_core.messages import AIMessage, HumanMessage, ToolMessage


async def ask(message, thread_id: str, show_tools: bool = True) -> str:
    # Send a message to the agent and display the tool calls plus the final answer
    messages = [HumanMessage(message)] if isinstance(message, str) else message
    result = await agent.ainvoke(
        {"messages": messages},
        config={"configurable": {"thread_id": thread_id}},
    )

    # Messages produced in this turn = everything after the last HumanMessage
    turn = []
    for msg in reversed(result["messages"]):
        if isinstance(msg, HumanMessage):
            break
        turn.insert(0, msg)

    if show_tools:
        for msg in turn:
            if isinstance(msg, AIMessage):
                for call in msg.tool_calls:
                    args = ", ".join(f"{k}={v!r}" for k, v in call["args"].items())
                    print(f"🔧 {call['name']}({args})")
            elif isinstance(msg, ToolMessage):
                icon = "❌" if msg.status == "error" else "✅"
                print(f"   {icon} {msg.name} returned {len(str(msg.content))} chars")

    answer = result["messages"][-1].content
    display(Markdown(f"**🤖 Ticket Desk:**\n\n{answer}"))
    return answer

## 5. Use case 1: Help-desk intake

A user reports a problem in plain English. The agent must:

1. infer the **product family** and **priority**,
2. call **`register_ticket`**,
3. reply with the new ticket ID.

We then ask a follow-up **in the same thread** to show that the agent remembers the conversation.

In [ ]:
answer = await ask(
    "Hi, this is Ramkumar from the Chennai office. Since 9 AM nobody on the 3rd floor has network access. "
    "All the lights on the floor switch are blinking amber.",
    thread_id="intake-ramkumar",
)

In [ ]:
# Memory: the agent remembers which ticket it just created
answer = await ask("Why did you choose that priority? And which product family did you file it under?", thread_id="intake-ramkumar")

In [ ]:
# A second, less urgent request in a NEW thread (separate conversation)
answer = await ask(
    "Hello, Kavya here. Could someone schedule a firmware upgrade on the Kochi SD-WAN edge next week? "
    "No issues right now, it's just routine maintenance.",
    thread_id="intake-kavya",
)

## 6. Use case 2: Triage & work a ticket

An engineer works on the most urgent ticket, from start to finish, in a single conversation:

1. **Find** open P1 tickets (tool: `search_tickets`).
2. **Triage** one of them using the server's **`triage_ticket` MCP prompt**.
3. **Start work**: move it to *In Progress* (tool: `update_ticket_status`).
4. **Resolve** it with remarks (tool: `resolve_ticket`).
5. **Verify** the result by reading the **`tickets://{ticket_id}` resource**.

In [ ]:
answer = await ask(
    "Show me all Open P1 tickets as a short table (ID, title, product family, age in days).",
    thread_id="triage",
)

# Pick the first ticket ID the agent mentioned
ticket_ids = re.findall(r"TKT-\d{5}", answer)
assert ticket_ids, "No open P1 tickets found - register one in use case 1 first."
TICKET_ID = ticket_ids[0]
print(f"\n👉 Selected for triage: {TICKET_ID}")

In [ ]:
# Load the server-side prompt template and send it into the SAME thread
triage_prompt = await client.get_prompt("tickets", "triage_ticket", arguments={"ticket_id": TICKET_ID})
answer = await ask(triage_prompt, thread_id="triage")

In [ ]:
# Memory: "this ticket" refers back to the ticket being triaged
answer = await ask(
    "Good analysis. Please move this ticket to In Progress with the note 'Assigned to Network L2 team'.",
    thread_id="triage",
)

In [ ]:
answer = await ask(
    "Update: the team fixed it. Please resolve the ticket. Resolved by Priya Nair. "
    "Remarks: root cause identified and fixed as per triage; service restored and verified with users.",
    thread_id="triage",
)

In [ ]:
# Verify straight from the MCP resource, without going through the LLM
blob = (await client.get_resources("tickets", uris=[f"tickets://{TICKET_ID}"]))[0]
ticket = json.loads(blob.as_string())
print(json.dumps({k: ticket[k] for k in ("ticket_id", "status", "resolved_by", "resolved_date")}, indent=2))
print("\nRemarks history:\n" + (ticket["remarks"] or ""))

## 7. Use case 3: Manager status report

The IT manager wants an executive summary. The **application** gathers the context:

- the **`tickets://stats` resource** (live KPIs),
- the **`summarize_open` prompt** (the list of unresolved tickets plus report instructions),

and hands both to the agent. The agent can still call **tools** for anything extra it needs.

In [ ]:
stats_blob = (await client.get_resources("tickets", uris=["tickets://stats"]))[0]
summary_prompt = await client.get_prompt("tickets", "summarize_open")

report_request = HumanMessage(
    "I am the IT Operations Manager. Prepare my weekly ticket status report.\n\n"
    f"### Live KPIs (resource tickets://stats)\n```json\n{stats_blob.as_string()}\n```\n\n"
    f"### Unresolved tickets (prompt summarize_open)\n{summary_prompt[0].content}\n\n"
    "Format the report with: KPI snapshot, executive summary, P1/P2 risks, "
    "and 3 recommended actions. You may use the tools for extra detail "
    "(for example, tickets resolved in the last 30 days)."
)

answer = await ask([report_request], thread_id="manager")

In [ ]:
# Memory: a follow-up question on the same report
answer = await ask("Which product family needs the most attention next week, and why? Answer in 3 bullets.", thread_id="manager")

## 8. Wrap-up

| What we did | How |
|---|---|
| Connected LangChain to a remote MCP server | `MultiServerMCPClient` over **Streamable HTTP** |
| Gave the LLM real capabilities | MCP **tools** become LangChain tools (`get_tools()`) |
| Provided application context | MCP **resources** (`get_resources()`) |
| Reused server-side prompt templates | MCP **prompts** (`get_prompt()`) |
| Customised behaviour | **System prompt** with business rules |
| Kept conversations coherent | **Memory** with `InMemorySaver` and `thread_id` |

**Try next**

- Ask the agent to *reopen* a resolved ticket, or to close one.
- Start a new `thread_id` and see that the agent has no memory of earlier threads.
- Stop the server and see how the agent fails. Then restart it.
- Add a second MCP server to `MultiServerMCPClient` and combine tools from both.